# DLIS Unit II: A1-A2-A3 CIFAR-10 Image Classification Challenge

This notebook completes the supplied starter notebook according to the handout.

**Run this notebook in Kaggle with a GPU enabled.** The notebook must be executed there so that the required validation metrics, timings, plots, model summaries, and optional ResNet50 weight download are visible before exporting the notebook PDF.

**Important:** use only the supplied competition data. Do not use the hidden test labels or any external CIFAR-10 copy for model selection.


## 0. Setup and reproducibility

Required protocol:
- seed = 42
- one fixed 80:20 stratified train/validation split
- batch size = 64
- metric = accuracy
- sparse categorical cross-entropy
- early stopping on `val_loss` with best-weight restoration for final A1/A2/A3
- A1: max 5 epochs
- A2 tuning: 4,000-image stratified subset, 2 folds, exactly 4 prescribed configurations, exactly 2 epochs per fold
- A2 final: max 6 epochs
- A3: frozen ImageNet ResNet50, 96x96 inputs, max 3 epochs


In [4]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


ModuleNotFoundError: No module named 'tensorflow'

## 1. Load, inspect, and split

The supplied package contains:
- `train.npz`: 15,000 labelled 32x32 RGB images
- `test.npz`: 3,000 test images with hidden labels
- `sample_submission.csv`
- `class_names.csv`


In [5]:
train_files = list(Path("/kaggle/input").glob("**/train.npz"))
if not train_files:
    raise FileNotFoundError(
        "train.npz was not found. Attach the competition data to this notebook."
    )

COMP_PATH = train_files[0].parent
print("Using competition data from:", COMP_PATH)

train = np.load(COMP_PATH / "train.npz")
test = np.load(COMP_PATH / "test.npz")

x_all, y_all = train["images"], train["labels"]
train_ids = train["ids"]

x_test, test_ids = test["images"], test["ids"]
class_names = pd.read_csv(COMP_PATH / "class_names.csv")["class_name"].tolist()
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

print("x_all shape:", x_all.shape)
print("y_all shape:", y_all.shape)
print("x_test shape:", x_test.shape)
print("train dtype:", x_all.dtype)
print("pixel range:", x_all.min(), "to", x_all.max())
print("labels shape:", y_all.shape)
print("unique labels:", np.unique(y_all))
print("sample_submission shape:", sample_submission.shape)
print("class names:", class_names)


FileNotFoundError: train.npz was not found. Attach the competition data to this notebook.

In [ ]:
# Class distribution
class_counts = pd.Series(y_all).value_counts().sort_index()
distribution_df = pd.DataFrame({
    "label": class_counts.index,
    "class_name": [class_names[i] for i in class_counts.index],
    "count": class_counts.values
})
display(distribution_df)

plt.figure(figsize=(10, 4))
sns.barplot(data=distribution_df, x="class_name", y="count")
plt.title("CIFAR-10 Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# At least 12 labelled images
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(x_all), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(x_all[idx])
    ax.set_title(f"{y_all[idx]} - {class_names[y_all[idx]]}")
    ax.axis("off")
fig.suptitle("12 Labelled Training Images", fontsize=14)
plt.tight_layout()
plt.show()


### Label format and loss choice

The labels are sparse integer class IDs from 0 through 9 rather than one-hot vectors. Therefore `sparse_categorical_crossentropy` is appropriate because it accepts integer class labels directly and the final layer produces a 10-class softmax probability distribution.


In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_all,
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

print("Final training split:", x_train.shape, y_train.shape)
print("Final validation split:", x_val.shape, y_val.shape)


## 2. Shared helpers

The timing below covers the model training call only, as required. Final models use early stopping on validation loss with restoration of the best weights.


In [ ]:
results = []

def train_and_record(name, model, xtr, ytr, xv, yv, epochs):
    early = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )

    start = time.perf_counter()
    history = model.fit(
        xtr, ytr,
        validation_data=(xv, yv),
        epochs=epochs,
        batch_size=64,
        callbacks=[early],
        verbose=1
    )
    elapsed = time.perf_counter() - start

    values = model.evaluate(xv, yv, verbose=0)
    history_df = pd.DataFrame(history.history)

    results.append({
        "model": name,
        "total_params": model.count_params(),
        "trainable_params": int(sum(np.prod(v.shape) for v in model.trainable_weights)),
        "time_seconds": elapsed,
        "epochs": len(history.history["loss"]),
        "best_val_loss": float(history_df["val_loss"].min()),
        "best_val_accuracy": float(history_df["val_accuracy"].max()),
        "final_val_accuracy": float(values[1])
    })
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["loss"], label="Train")
    axes[0].plot(history.history["val_loss"], label="Validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train")
    axes[1].plot(history.history["val_accuracy"], label="Validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 3. A1 - Deep Dense baseline

Architecture required by the handout:
`Rescaling -> Flatten -> Dense(256, ReLU) -> Dense(128, ReLU) -> Dropout(0.2) -> Dense(10, softmax)`

The two hidden Dense layers are exactly 256 and 128 units. The output Dense layer is the required 10-class classifier.


In [ ]:
def build_a1():
    tf.keras.backend.clear_session()

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(32, 32, 3)),
        tf.keras.layers.Rescaling(1.0 / 255.0),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

a1 = build_a1()
a1.summary()

history_a1 = train_and_record(
    "A1", a1,
    x_train, y_train,
    x_val, y_val,
    epochs=5
)
plot_history(history_a1, "A1")


### A1 reasoning prompt

Because the image is flattened before the first Dense layer, each of the 3,072 input pixel values connects directly to each of the 256 first-layer neurons. This creates a very large parameter count in the first Dense layer and discards explicit 2D spatial locality, motivating the CNN comparison in A2.


## 4. A2 - Multi-convolution CNN

Architecture:
`Rescaling -> [Conv2D -> ReLU -> MaxPool] x 3 -> Flatten -> Dense(128, ReLU) -> Dropout -> Dense(10, softmax)`

The convolutional filter counts, learning rate, and dropout are controlled by the required grid.


In [ ]:
def build_a2(filters=(16, 32, 64), lr=1e-3, dropout=0.2):
    tf.keras.backend.clear_session()

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(32, 32, 3)),
        tf.keras.layers.Rescaling(1.0 / 255.0),

        tf.keras.layers.Conv2D(filters[0], (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(filters[1], (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(filters[2], (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


### 4.1 Mandatory small grid-search CV

Exactly:
- 4,000 stratified tuning images
- 2-fold `StratifiedKFold`
- 4 prescribed configurations
- 2 epochs per fold
- fresh model for every fold/configuration
- record best fold validation accuracy and elapsed time
- summarize mean and standard deviation


In [ ]:
tune_x, _, tune_y, _ = train_test_split(
    x_train,
    y_train,
    train_size=4000,
    random_state=42,
    stratify=y_train
)

configs = [
    {"name": "G1", "filters": (16, 32, 64), "lr": 1e-3, "dropout": 0.2},
    {"name": "G2", "filters": (16, 32, 64), "lr": 3e-4, "dropout": 0.3},
    {"name": "G3", "filters": (32, 64, 128), "lr": 1e-3, "dropout": 0.2},
    {"name": "G4", "filters": (32, 64, 128), "lr": 3e-4, "dropout": 0.3},
]

skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
grid_rows = []

for cfg in configs:
    for fold, (tr_idx, va_idx) in enumerate(skf.split(tune_x, tune_y), start=1):
        tf.keras.backend.clear_session()
        model = build_a2(
            filters=cfg["filters"],
            lr=cfg["lr"],
            dropout=cfg["dropout"]
        )

        start = time.perf_counter()
        hist = model.fit(
            tune_x[tr_idx], tune_y[tr_idx],
            validation_data=(tune_x[va_idx], tune_y[va_idx]),
            epochs=2,
            batch_size=64,
            verbose=1
        )
        elapsed = time.perf_counter() - start

        grid_rows.append({
            "config": cfg["name"],
            "filters": str(cfg["filters"]),
            "learning_rate": cfg["lr"],
            "dropout": cfg["dropout"],
            "fold": fold,
            "params": model.count_params(),
            "time_seconds": elapsed,
            "best_fold_val_accuracy": max(hist.history["val_accuracy"])
        })

grid_results = pd.DataFrame(grid_rows)
display(grid_results)

grid_summary = (
    grid_results
    .groupby(["config", "filters", "learning_rate", "dropout", "params"], as_index=False)
    .agg(
        mean_val_accuracy=("best_fold_val_accuracy", "mean"),
        std_val_accuracy=("best_fold_val_accuracy", "std"),
        mean_time_seconds=("time_seconds", "mean")
    )
    .sort_values(
        ["mean_val_accuracy", "params", "mean_time_seconds"],
        ascending=[False, True, True]
    )
)

display(grid_summary)


### 4.2 Select and train the final A2

Selection rule: highest mean CV accuracy; ties are broken by fewer parameters, then shorter mean time.

The selected configuration is then rebuilt from scratch and trained on the fixed 80:20 final training/validation split for at most 6 epochs.


In [ ]:
best_row = grid_summary.iloc[0]

best_config = {
    "filters": tuple(int(x.strip()) for x in best_row["filters"].strip("()").split(",")),
    "lr": float(best_row["learning_rate"]),
    "dropout": float(best_row["dropout"])
}

print("Selected A2 configuration:", best_config)
print("Selection basis: highest mean CV accuracy, then fewer parameters, then shorter mean time.")

a2 = build_a2(**best_config)
a2.summary()

history_a2 = train_and_record(
    "A2", a2,
    x_train, y_train,
    x_val, y_val,
    epochs=6
)
plot_history(history_a2, "A2")


## 5. A3 - Frozen ImageNet ResNet50

Required preprocessing is different from A1/A2:
`32x32x3 -> resize to 96x96 -> ResNet preprocess_input -> frozen ResNet50(pooling='avg') -> Dropout(0.3) -> Dense(10, softmax)`

Do **not** apply `Rescaling(1/255)` to A3. The ResNet preprocessing path expects its own preprocessing operation.


In [ ]:
def build_a3():
    tf.keras.backend.clear_session()

    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Resizing(96, 96)(inputs)
    x = tf.keras.layers.Lambda(
        tf.keras.applications.resnet.preprocess_input,
        name="resnet_preprocess"
    )(x)

    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(96, 96, 3)
    )
    base.trainable = False

    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="A3_Frozen_ResNet50")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

a3 = build_a3()
a3.summary()

history_a3 = train_and_record(
    "A3", a3,
    x_train, y_train,
    x_val, y_val,
    epochs=3
)
plot_history(history_a3, "A3")


## 6. Final evidence and reasoning

The final comparison table includes A1, the selected A2, and A3. The best validation model is selected only from validation evidence, never from the hidden test set.


In [ ]:
comparison = pd.DataFrame(results).copy()

display(comparison.style.format({
    "total_params": "{:,}",
    "trainable_params": "{:,}",
    "time_seconds": "{:.2f}",
    "epochs": "{:.0f}",
    "best_val_loss": "{:.4f}",
    "best_val_accuracy": "{:.4f}",
    "final_val_accuracy": "{:.4f}"
}))

# Validation-only model selection for the final evidence section.
comparison_ranked = comparison.sort_values(
    ["best_val_accuracy", "final_val_accuracy", "trainable_params", "time_seconds"],
    ascending=[False, False, True, True]
)

best_model_name = comparison_ranked.iloc[0]["model"]
model_map = {"A1": a1, "A2": a2, "A3": a3}
best_model = model_map[best_model_name]

print("Best validation model:", best_model_name)


In [ ]:
# Confusion matrix for the best validation model
val_prob = best_model.predict(x_val, batch_size=64, verbose=0)
val_pred = np.argmax(val_prob, axis=1)

cm = confusion_matrix(y_val, val_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()

print(classification_report(
    y_val,
    val_pred,
    target_names=class_names,
    digits=4
))


In [ ]:
# At least 12 validation predictions, including misclassified examples where possible.
correct_idx = np.where(val_pred == y_val)[0]
wrong_idx = np.where(val_pred != y_val)[0]

rng = np.random.default_rng(42)

n_wrong = min(len(wrong_idx), 6)
n_correct = min(len(correct_idx), 6)

selected_wrong = rng.choice(wrong_idx, size=n_wrong, replace=False) if n_wrong else np.array([], dtype=int)
selected_correct = rng.choice(correct_idx, size=n_correct, replace=False) if n_correct else np.array([], dtype=int)

selected = np.concatenate([selected_wrong, selected_correct])

# Fill to 12 total when necessary.
if len(selected) < 12:
    remaining = np.setdiff1d(np.arange(len(x_val)), selected)
    extra = rng.choice(remaining, size=min(12-len(selected), len(remaining)), replace=False)
    selected = np.concatenate([selected, extra])

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, idx in zip(axes.flat, selected[:12]):
    true_label = y_val[idx]
    pred_label = val_pred[idx]
    ax.imshow(x_val[idx])
    marker = "Correct" if true_label == pred_label else "MISCLASSIFIED"
    ax.set_title(
        f"{marker}\nT: {class_names[true_label]} | P: {class_names[pred_label]}"
    )
    ax.axis("off")

fig.suptitle(f"Validation Predictions - {best_model_name}", fontsize=14)
plt.tight_layout()
plt.show()


### Evidence-based discussion

Write your final discussion after running the notebook, using the measured outputs rather than assumptions. Cover:
- parameter efficiency;
- training time;
- train/validation curve behaviour and overfitting/underfitting evidence;
- why convolutional parameter sharing and local receptive fields change A2 relative to A1;
- what a frozen pretrained ResNet50 contributes;
- limitations: short epoch budgets, compact dataset, one main split, and unequal preprocessing/computation cost for A3.


## 7. Formative Kaggle submission

This CSV is for leaderboard feedback only. Use the model chosen from validation evidence above.


In [ ]:
# Use the best validation model selected above.
best_model = model_map[best_model_name]

test_probabilities = best_model.predict(x_test, batch_size=64, verbose=1)
predicted_labels = np.argmax(test_probabilities, axis=1)

submission = pd.DataFrame({
    "id": test_ids,
    "label": predicted_labels
})

# Required validation checks
assert list(submission.columns) == ["id", "label"]
assert len(submission) == len(sample_submission)
assert len(submission) == len(test_ids)
assert submission["id"].isna().sum() == 0
assert submission["label"].isna().sum() == 0
assert submission["label"].between(0, 9).all()
assert submission["id"].is_unique
assert submission["id"].tolist() == sample_submission["id"].tolist()

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
display(submission.head(10))


## 8. Export checklist

Before exporting the complete notebook as PDF, confirm that the executed notebook visibly contains:
- GPU and TensorFlow version;
- array shapes, dtype, pixel range, class counts, and 12 labelled images;
- the fixed stratified split and seed;
- A1 summary, parameter counts, timing, epochs, metrics, and curves;
- all 8 A2 CV fold runs plus the mean/std grid summary and selected configuration;
- selected A2 final summary, timing, metrics, and curves;
- A3 frozen ResNet50 summary, timing, metrics, and curves;
- final comparison table;
- confusion matrix and 12 validation predictions including errors;
- evidence-based reasoning and limitations;
- test prediction creation and submission validation.

Then use the browser's **Print -> Save as PDF** to export the complete notebook.
